# Gold Bridge: Station to Line

Build the many-to-many relationship between canonical Tube stations and Tube lines.

This notebook:

1. Reads Silver station-line relationships.
2. Resolves StopPoint IDs to canonical station IDs.
3. Joins current Gold station versions.
4. Joins current Gold line versions.
5. Validates referential integrity.
6. Writes the Gold bridge table.

**Sources:**

- `workspace.urbanpulse_silver.tfl_stop_point_lines`
- `workspace.urbanpulse_silver.tfl_stop_points`
- `workspace.urbanpulse_gold.dim_station`
- `workspace.urbanpulse_gold.dim_line`

**Target:** `workspace.urbanpulse_gold.bridge_station_line`

**Grain:** One current station-to-line relationship.

## 1. Initialise project paths  

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import transformation components

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.bridge_station_line import (
    prepare_station_line_bridge,
)

## 3. Define source and target tables

In [0]:
SILVER_STATION_LINES = (
    "workspace."
    "urbanpulse_silver."
    "tfl_stop_point_lines"
)

SILVER_STOP_POINTS = (
    "workspace."
    "urbanpulse_silver."
    "tfl_stop_points"
)

DIM_STATION = (
    "workspace."
    "urbanpulse_gold."
    "dim_station"
)

DIM_LINE = (
    "workspace."
    "urbanpulse_gold."
    "dim_line"
)

TARGET_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "bridge_station_line"
)

In [0]:
station_lines_df = spark.table(
    SILVER_STATION_LINES
)

stop_points_df = spark.table(
    SILVER_STOP_POINTS
)

dim_station_df = spark.table(
    DIM_STATION
)

dim_line_df = spark.table(
    DIM_LINE
)

## 5. Resolve station and line relationships

Map Silver StopPoint relationships to canonical station IDs, then resolve the current Gold surrogate keys.

In [0]:
bridge_df = (
    prepare_station_line_bridge(
        station_lines_df=station_lines_df,
        stop_points_df=stop_points_df,
        dim_station_df=dim_station_df,
        dim_line_df=dim_line_df,
    )
)

print(
    f"Bridge relationships: "
    f"{bridge_df.count()}"
)

display(
    bridge_df
    .orderBy(
        "station_id",
        "line_id",
    )
)

## 6. Create the bridge key

Each station-line relationship receives a deterministic Gold key based on the current station and line dimension versions.

In [0]:
bridge_df = (
    bridge_df
    .withColumn(
        "station_line_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("station_key"),
                F.col("line_key"),
            ),
            256,
        ),
    )
    .withColumn(
        "created_at",
        F.current_timestamp(),
    )
    .select(
        "station_line_key",
        "station_key",
        "line_key",
        "station_id",
        "line_id",
        "source_snapshot_at",
        "created_at",
    )
)

## 7. Validate bridge uniqueness

Each current station-to-line relationship must appear exactly once.

In [0]:
duplicate_relationships_df = (
    bridge_df
    .groupBy(
        "station_id",
        "line_id",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_keys_df = (
    bridge_df
    .groupBy(
        "station_line_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_relationships_df.count() > 0:
    display(
        duplicate_relationships_df
    )

    raise ValueError(
        "Duplicate station-line "
        "relationships detected."
    )

if duplicate_keys_df.count() > 0:
    display(
        duplicate_keys_df
    )

    raise ValueError(
        "Duplicate station_line_key "
        "values detected."
    )

print(
    "Bridge uniqueness validation passed."
)

## 8. Validate dimension references

Every bridge relationship must reference a valid current station dimension row and a valid current line dimension row.

In [0]:
missing_station_refs_df = (
    bridge_df.alias("bridge")
    .join(
        dim_station_df
        .filter(
            F.col("is_current")
        )
        .select(
            "station_key"
        )
        .alias("station"),
        on="station_key",
        how="left_anti",
    )
)

missing_line_refs_df = (
    bridge_df.alias("bridge")
    .join(
        dim_line_df
        .filter(
            F.col("is_current")
        )
        .select(
            "line_key"
        )
        .alias("line"),
        on="line_key",
        how="left_anti",
    )
)

missing_station_refs = (
    missing_station_refs_df.count()
)

missing_line_refs = (
    missing_line_refs_df.count()
)

if missing_station_refs > 0:
    display(
        missing_station_refs_df
    )

    raise ValueError(
        f"{missing_station_refs} bridge "
        "rows have invalid station keys."
    )

if missing_line_refs > 0:
    display(
        missing_line_refs_df
    )

    raise ValueError(
        f"{missing_line_refs} bridge "
        "rows have invalid line keys."
    )

print(
    "Bridge referential integrity passed."
)

## 9. Validate source relationship coverage

Compare canonical Silver relationships with the relationships successfully resolved into Gold.

Any missing relationships require investigation before the bridge is published.

In [0]:
latest_relationship_snapshot = (
    station_lines_df
    .agg(
        F.max(
            "snapshot_at"
        ).alias(
            "latest_snapshot"
        )
    )
    .first()["latest_snapshot"]
)

latest_stop_point_snapshot = (
    stop_points_df
    .agg(
        F.max(
            "snapshot_at"
        ).alias(
            "latest_snapshot"
        )
    )
    .first()["latest_snapshot"]
)

In [0]:
canonical_silver_relationships_df = (
    station_lines_df
    .filter(
        F.col("snapshot_at")
        == latest_relationship_snapshot
    )
    .select(
        "stop_point_id",
        "line_id",
    )
    .dropDuplicates()

    .join(
        stop_points_df
        .filter(
            F.col("snapshot_at")
            == latest_stop_point_snapshot
        )
        .filter(
            F.col(
                "station_naptan"
            ).isNotNull()
        )
        .select(
            "stop_point_id",
            F.col(
                "station_naptan"
            ).alias(
                "station_id"
            ),
        )
        .dropDuplicates(),
        on="stop_point_id",
        how="inner",
    )

    # Restrict validation to Tube lines
    # represented in dim_line.
    .join(
        dim_line_df
        .filter(
            F.col("is_current")
        )
        .select("line_id")
        .distinct(),
        on="line_id",
        how="inner",
    )

    .select(
        "station_id",
        "line_id",
    )
    .dropDuplicates()
)

## Inspect out-of-scope transport relationships

TfL StopPoint data can include bus and other interchange services.

UrbanPulse currently models London Underground lines only, so relationships to services outside `dim_line` are excluded from the Gold Tube bridge. They are retained in Silver and can be incorporated later if the project scope expands.

In [0]:
unresolved_relationships_df = (
    canonical_silver_relationships_df
    .join(
        bridge_df.select(
            "station_id",
            "line_id",
        ),
        on=[
            "station_id",
            "line_id",
        ],
        how="left_anti",
    )
)

unresolved_count = (
    unresolved_relationships_df.count()
)

if unresolved_count > 0:
    display(
        unresolved_relationships_df
    )

    raise ValueError(
        f"{unresolved_count} canonical "
        "station-line relationships "
        "could not be resolved to Gold."
    )

print(
    "All canonical relationships "
    "resolved successfully."
)

## 10. Write the Gold bridge

The bridge represents the current station-line relationship state and is rebuilt on each successful execution.

In [0]:
(
    bridge_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Created Gold bridge: "
    f"{TARGET_TABLE}"
)

## 11. Verify station-line relationships

In [0]:
%sql
SELECT
    station_line_key,
    station_id,
    line_id,
    station_key,
    line_key,
    source_snapshot_at
FROM workspace.urbanpulse_gold.bridge_station_line
ORDER BY station_id, line_id;

In [0]:
%sql
SELECT
    s.station_name,
    l.line_name,
    b.station_id,
    b.line_id
FROM workspace.urbanpulse_gold.bridge_station_line b

INNER JOIN workspace.urbanpulse_gold.dim_station s
    ON b.station_key = s.station_key

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON b.line_key = l.line_key

ORDER BY
    s.station_name,
    l.line_name;

In [0]:
%sql
SELECT
    s.station_name,
    COUNT(DISTINCT b.line_key) AS line_count,
    COLLECT_SET(
        l.line_name
    ) AS lines
FROM workspace.urbanpulse_gold.bridge_station_line b

INNER JOIN workspace.urbanpulse_gold.dim_station s
    ON b.station_key = s.station_key

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON b.line_key = l.line_key

GROUP BY
    s.station_key,
    s.station_name

HAVING COUNT(DISTINCT b.line_key) > 1

ORDER BY
    line_count DESC,
    s.station_name;

In [0]:
%sql
SELECT
    l.line_name,
    COUNT(
        DISTINCT b.station_key
    ) AS stations
FROM workspace.urbanpulse_gold.bridge_station_line b

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON b.line_key = l.line_key

GROUP BY
    l.line_key,
    l.line_name

ORDER BY stations DESC;

In [0]:
%sql
SELECT b.*
FROM workspace.urbanpulse_gold.bridge_station_line b

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_station s
    ON b.station_key = s.station_key;

In [0]:
%sql
SELECT b.*
FROM workspace.urbanpulse_gold.bridge_station_line b

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_line l
    ON b.line_key = l.line_key;

In [0]:
%sql
SELECT
    b.station_id,
    b.line_id
FROM workspace.urbanpulse_gold.bridge_station_line b

INNER JOIN workspace.urbanpulse_gold.dim_station s
    ON b.station_key = s.station_key

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON b.line_key = l.line_key

WHERE
    s.is_current = FALSE
    OR l.is_current = FALSE;

In [0]:
%sql
SELECT COUNT(*) AS relationships
FROM workspace.urbanpulse_gold.bridge_station_line;